# Ingestion: dev exploration

This notebook runs the ingestion module (`comps.ingest`) against a local data folder and looks at what it produced.

- Set `DATA_DIR` in the environment, or edit the default below. The folder must contain `companies.json` and `fx_rates.json`.
- **Clear all outputs before committing.** They would contain the supplied data, which must not go into the repo.

Setup: `uv pip install -e '.[notebook]'`, then `jupyter lab scripts/dev/ingestion.ipynb`.

In [ ]:
import json
import os
from dataclasses import asdict
from pathlib import Path

import pandas as pd

from comps.config import IngestConfig
from comps.ingest import DatasetError, build_dataset, load_dataset

pd.set_option("display.max_colwidth", 120)
DATA_DIR = Path(os.environ.get("DATA_DIR", "~/Downloads/genios-takehome/data")).expanduser()
ds = load_dataset(DATA_DIR, IngestConfig())

print(f"dataset version : {ds.version}")
print(f"reference date  : {ds.reference_date}")
print(f"records         : {ds.record_count}")
print(f"entities        : {len(ds.entities)}  (keyed by COUNTRY:TICKER)")
print(f"fx rates        : {dict(ds.fx_rates)}")
for f in ds.files:
    print(f"file            : {f.name} sha256={f.sha256[:16]}…")

## 1. Findings overview
Counts by code and severity. A `derived` source would mean model-derived; at this stage every finding comes from validation.

In [ ]:
findings = pd.DataFrame([asdict(f) for f in ds.all_findings()])
(
    findings.groupby(["severity", "code", "source"])
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["severity", "count"], ascending=[True, False])
)

## 2. All findings except info

In [ ]:
cols = ["severity", "code", "entity_keys", "fields", "affects", "message"]
findings[findings.severity != "info"][cols].sort_values(["severity", "code"])

## 3. Entity resolution
Companies built from more than one record, through exact duplicates, same-date conflicts or a time series.

In [ ]:
rows = []
for key, e in ds.entities.items():
    if len(e.record_ids) > 1 or len(e.aliases) > 1:
        rows.append(
            {
                "key": key,
                "name": e.name,
                "aliases": e.aliases,
                "observations": [(o.record_id, str(o.price_date)) for o in e.observations],
                "superseded": [o.record_id for o in e.superseded],
                "findings": [f.code for f in e.findings],
            }
        )
pd.DataFrame(rows)

## 4. Possible same issuer
Different keys with identical content. They're kept as separate companies and later excluded from each other's peer lists.

In [ ]:
groups = {f.entity_keys for f in ds.all_findings() if f.code == "possible_same_issuer"}
pd.DataFrame([{"keys": g, "names": [ds.entities[k].name for k in g]} for g in sorted(groups)])

## 5. Currency and unit conversion
Every record that isn't plain USD in millions, so the USD normalization can be checked by hand.

In [ ]:
rows = []
for key, e in ds.entities.items():
    o, src = e.latest, e.latest.original
    if src.get("price_currency") != "USD" or src.get("reporting_currency") != "USD" or src.get("units") != "millions":
        rows.append(
            {
                "key": key,
                "price": src.get("price"),
                "price_ccy": src.get("price_currency"),
                "price_fx": o.conversion.price_fx,
                "price_usd": o.usd.price,
                "rep_ccy": src.get("reporting_currency"),
                "units": src.get("units"),
                "reporting_fx": o.conversion.reporting_fx,
                "unit_factor": o.conversion.unit_factor,
                "revenue": src.get("revenue"),
                "revenue_usd_m": o.usd.revenue,
            }
        )
pd.DataFrame(rows)

## 6. Inspect one company
Change `KEY` to any `COUNTRY:TICKER`.

In [ ]:
KEY = next(iter(ds.entities))
e = ds.entities[KEY]
print(e.key, "|", e.name, "|", e.sector, ">", e.industry, ">", e.sub_industry)
print(e.description)
money = ("total_debt", "cash", "revenue", "ebitda", "net_income")
pd.DataFrame(
    {
        "original": {k: e.latest.original.get(k) for k in ("price", "shares_outstanding") + money},
        "usd": {
            "price": e.latest.usd.price,
            "shares_outstanding": e.latest.shares_outstanding,
            **{k: getattr(e.latest.usd, k) for k in money},
        },
    }
)

## 7. Blocking errors leave nothing half-built
A malformed FX file raises `DatasetError`. The caller keeps the previous dataset.

In [ ]:
companies = json.loads((DATA_DIR / "companies.json").read_text())

try:
    build_dataset(companies, {"base": "EUR", "rates": {}})
except DatasetError as exc:
    print("blocked:", exc)